# 云运维故障演练 Notebook
## 从「看不懂告警」到「独立值班」的 9 个递进实验

配套项目：`cloud-support-lab`。所有指标、日志都来自本机真实运行，仓库里没有写死的观测值。

---

### 这个 Notebook 解决什么问题

README 告诉你「怎么把服务跑起来」，但排障能力不是跑起来就会的。真实值班需要一条固定动作链：

> 看见现象 → 判断影响面 → 分层定位 → 拿证据 → 先止损 → 再根治 → 复盘沉淀

本 Notebook 把这条链拆成 9 个实验，每个实验都让你**亲手注入故障、亲眼看到指标、自己下判断**，
最后对照参考结论检查思路，而不是背答案。

### 你会得到什么

| 产出 | 内容 |
|---|---|
| 9 个可重复实验 | 1 个健康基线 + 8 个故障场景（接入层 / 网络 / 应用 / 数据库 / 缓存 / 存储 / 容器） |
| 一套定位方法 | RED + USE 两套指标视角、三层架构定位、8 场景鉴别诊断表 |
| 一套值班物料 | 工单模板、RCA 模板、客户沟通话术、自评评分卡 |
| 面试素材 | 每个场景的「一句话结论 + 追问应对」 |

### 使用方法

1. 需要 Python 3.9+。仓库本身只用标准库；Notebook 需要 Jupyter：
   ```bash
   python3 -m pip install notebook    # 或 jupyterlab；VS Code 装 Jupyter 扩展也可以
   ```
2. 打开终端，进入仓库后启动：
   ```bash
   cd cloud-support-lab
   jupyter notebook lab/notebooks/cloud-fault-drill.ipynb
   ```
3. **从上到下依次运行**。每个 Lab 都先自己回答「观察任务」，再看「参考结论」。
4. 任何单元格中断、端口占用或进程残留：先运行工具箱里的 `stop_app()`。
5. Docker 不是必须的。想练「真实容器 + 真实 SLB 502」，看最后的 **Path B** 一节。

> 说明：本环境里的故障是**确定性仿真**。`/api/order/list` 是真实 HTTP 接口，但资源类故障
> （CPU、慢 SQL、内存、热 key）主要体现为 `/metrics` 里的指标劣化；只有连接池、磁盘、DNS、
> 健康检查这几类会直接在接口上返回 5xx。每个 Lab 都会明确告诉你「该看接口还是看指标」。


## 学习路线：难度是设计过的

| # | 实验 | 层次 | 难度 | 核心能力 |
|---|---|---|---|---|
| Lab 0 | 健康基线 | 全链路 | ★☆☆ | 认识正常值，建立「基线感」 |
| Lab 1 | SLB 健康检查失败 | 接入层 | ★★☆ | 区分「应用 5xx」与「网关 5xx」 |
| Lab 2 | DNS 解析异常 | 网络层 | ★★☆ | 识别间歇性失败与超时量级 |
| Lab 3 | 连接池耗尽 | 应用↔DB | ★★☆ | 用 DB 指标区分「池满」与「DB 慢」 |
| Lab 4 | 慢 SQL / DB CPU | 数据层 | ★★★ | 从 P99 秒级反推到执行计划 |
| Lab 5 | 应用 CPU 打满 | 应用层 | ★★★ | 识别「资源饱和」的连锁假象 |
| Lab 6 | Redis 热 key | 缓存层 | ★★★ | 从命中率与回源压力定位热 key |
| Lab 7 | 磁盘 / inode 打满 | 基础设施 | ★★★ | 区分空间满与 inode 满、快失败 |
| Lab 8 | 内存泄漏 OOMKill | 容器生命周期 | ★★★★ | 看峰值/重启，而不是看均值 |

**建议节奏**：一次只做 1~2 个 Lab。做完后在纸上回答三个问题：
① 这个场景最先变的是哪个指标？② 它最容易和哪个场景混淆？③ 如果客户在催，你先做什么？


In [ ]:
import sys, shutil
from pathlib import Path

def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "lab" / "sim" / "app.py").exists():
            return p
    raise RuntimeError("找不到仓库根目录：请在 cloud-support-lab/ 或 lab/notebooks/ 下打开本 Notebook")

ROOT = find_root()
LAB = ROOT / "lab"
EVID = ROOT / "evidence"

print("Python   :", sys.version.split()[0])
print("仓库根目录:", ROOT)
print("app.py   :", "OK" if (LAB / "sim" / "app.py").exists() else "缺失")
print("runner.py:", "OK" if (LAB / "sim" / "runner.py").exists() else "缺失")
print("docker   :", shutil.which("docker") or "未安装（不影响路径 A）")
try:
    import matplotlib
    print("matplotlib:", matplotlib.__version__, "(图表可用)")
except ImportError:
    print("matplotlib: 未安装（本 Notebook 用文本/HTML 表格，不依赖它）")


## 实验方法：三层 + 两套指标视角

真实排障不是「凭感觉点 Grafana」，而是先固定观察框架。

### 三层架构（从外到内）

```text
客户端
  │
  ▼
接入层   SLB / CLB / nginx / HAProxy      ← 健康检查、502/504、连接数
  │
  ▼
应用层   Java/Go/Python 进程              ← 线程池、连接池、GC、延迟、5xx
  │
  ▼
数据层   MySQL / Redis                    ← 连接数、CPU、慢查询、命中率、热 key
  │
  ▼
基础设施 节点 / 磁盘 / 网络 / DNS          ← CPU、内存、磁盘、inode、解析
```

**定位原则**：先确认「错误发生在哪一层」，再往下钻。不要一上来就重启，也不要把所有指标一起看。

### 两套指标视角

| 视角 | 看什么 | 用在哪 |
|---|---|---|
| **RED** | Rate（请求量）、Errors（错误率）、Duration（P95/P99） | 判断「服务是否异常、异常形态是什么」 |
| **USE** | Utilization（使用率）、Saturation（饱和/排队）、Errors | 判断「哪个资源被打满、是否排队」 |

值班时的顺序通常是：**先 RED 确认现象，再 USE 找瓶颈，最后看日志拿证据**。

### 四步闭环

1. **界定影响面**：哪些接口/客户/地域？错误率多少？从几点开始？是否在扩大？
2. **分层定位**：接入层 → 应用层 → 数据层 → 基础设施，逐层排除。
3. **先止损**：回滚、摘除、扩容、限流、切换；需要保留现场时先取证再重启。
4. **再根治**：找到根因，补监控/告警/容量/变更约束，写 RCA。

> 三个不要：不要一上来重启（现场没了）；不要只看一个指标（容易误判）；不要对客户说「不是我们的问题」。


### 指标字典（先混个脸熟，后面每个 Lab 都会用到）

| 指标 | 所属层 | 含义 | 健康基线（本仿真） |
|---|---|---|---|
| `http_requests_total{status="2xx"}` | 应用 | 成功请求累计数 | 约 118 req/tick |
| `http_requests_total{status="5xx"}` | 应用 | 应用自己返回的 5xx | 0 |
| `http_request_duration_ms_p50/p95/p99` | 应用 | 请求延迟分位 | 14 / 179 / 200 ms |
| `app_worker_busy` | 应用 | 忙碌工作线程数 | 约 8 / 16 |
| `app_conn_pool_in_use` / `_size` | 应用 | 已借出的 DB 连接 / 池上限 | 约 6 / 10 |
| `db_connections_used` / `_max` | 数据 | DB 服务端连接数 / 上限 | 约 9 / 100 |
| `db_cpu_percent` | 数据 | 数据库 CPU | 22% |
| `db_slow_queries_total` | 数据 | 慢查询累计数 | 0 |
| `cache_latency_ms` | 数据 | 缓存 P95 延迟 | 约 4.4 ms |
| `cache_hit_ratio` | 数据 | 缓存命中率 | 约 0.85 |
| `node_cpu_percent` | 基础设施 | 节点/实例 CPU | 25% |
| `node_mem_percent` | 基础设施 | 节点/实例内存 | 42% |
| `node_disk_percent` | 基础设施 | 磁盘空间使用率 | 55% |
| `node_inode_percent` | 基础设施 | inode 使用率 | 30% |
| `container_restarts_total` | 容器 | 容器重启累计数 | 0 |
| `gateway_upstream_5xx_total` | 接入层 | 网关侧 5xx（502/504）累计 | 0 |

> 记不住没关系。方法是：**先看 RED 找异常，再看 USE 找资源，最后用日志确认**。


In [ ]:
import json, time, subprocess, sys, tempfile, http.client, urllib.parse, random, socket
from collections import Counter
from pathlib import Path
from IPython.display import display, HTML, Markdown

# 用 http.client 直连本机，彻底绕开 http_proxy/https_proxy（否则 127.0.0.1 也可能被代理）
STATE = {"proc": None, "port": None, "sid": None, "log": None}

LABELS = {
    "http_requests_total{status=\"2xx\"}": "2xx 累计",
    "http_requests_total{status=\"5xx\"}": "5xx 累计",
    "http_request_duration_ms_p50": "P50 (ms)",
    "http_request_duration_ms_p95": "P95 (ms)",
    "http_request_duration_ms_p99": "P99 (ms)",
    "app_worker_busy": "忙碌线程",
    "app_conn_pool_in_use": "池已用",
    "app_conn_pool_size": "池上限",
    "db_connections_used": "DB 连接",
    "db_connections_max": "DB 上限",
    "db_cpu_percent": "DB CPU%",
    "db_slow_queries_total": "慢查询累计",
    "node_cpu_percent": "节点 CPU%",
    "node_mem_percent": "节点内存%",
    "node_disk_percent": "磁盘%",
    "node_inode_percent": "inode%",
    "cache_latency_ms": "缓存 P95 (ms)",
    "cache_hit_ratio": "缓存命中率",
    "container_restarts_total": "重启累计",
    "gateway_upstream_5xx_total": "网关 5xx 累计",
}
REF = {
    "node_cpu_percent": "≈25%",
    "node_mem_percent": "≈42%",
    "node_disk_percent": "≈55%",
    "node_inode_percent": "≈30%",
    "db_cpu_percent": "≈22%",
    "app_conn_pool_in_use": "≈6 / 10",
    "app_worker_busy": "≈8 / 16",
    "http_request_duration_ms_p50": "≈14 ms",
    "http_request_duration_ms_p95": "≈179 ms",
    "http_request_duration_ms_p99": "≈200 ms",
    "cache_latency_ms": "≈4.4 ms",
    "cache_hit_ratio": "≈0.85",
    "http_requests_total{status=\"5xx\"}": "0",
    "gateway_upstream_5xx_total": "0（累计不增长）",
    "container_restarts_total": "0（不增长）",
}
BASELINE_KEYS = [
    "http_request_duration_ms_p50", "http_request_duration_ms_p95",
    "http_request_duration_ms_p99", "node_cpu_percent", "node_mem_percent",
    "node_disk_percent", "node_inode_percent", "db_cpu_percent",
    "app_conn_pool_in_use", "app_conn_pool_size",
    "cache_latency_ms", "cache_hit_ratio",
    "http_requests_total{status=\"5xx\"}",
]

display(HTML("<style>.lab-table{border-collapse:collapse;font-size:13px;margin:6px 0}"
             ".lab-table th,.lab-table td{border:1px solid #d9dee5;padding:4px 9px}"
             ".lab-table th{background:#f3f6fa;text-align:left}"
             ".lab-note{background:#fff8e6;border-left:4px solid #e6a23c;padding:8px 12px;margin:8px 0}"
             "</style>"))


def http_get(path, port=None, timeout=3.0):
    port = port or STATE["port"]
    target = str(path)
    host = "127.0.0.1"
    if target.startswith("http://"):
        u = urllib.parse.urlsplit(target)
        host = u.hostname or host
        port = u.port or 80
        target = u.path + (f"?{u.query}" if u.query else "")
    try:
        conn = http.client.HTTPConnection(host, int(port), timeout=timeout)
        conn.request("GET", target)
        resp = conn.getresponse()
        body = resp.read().decode("utf-8", "replace")
        status = resp.status
        conn.close()
        return status, body
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"


def stop_app():
    p = STATE.get("proc")
    if p and p.poll() is None:
        p.terminate()
        try:
            p.wait(timeout=3)
        except subprocess.TimeoutExpired:
            p.kill()
    log = STATE.get("log")
    if log:
        try:
            log.close()
        except Exception:
            pass
    STATE.update(proc=None, log=None)


def free_port():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
    s.close()
    return port


def start_app(sid=None, port=None, tick_period=0.1, boot_timeout=8):
    stop_app()
    port = port or free_port()
    cmd = [sys.executable, str(LAB / "sim" / "app.py"),
           "--port", str(port), "--tick-period", str(tick_period)]
    if sid:
        cmd += ["--scenario", sid]
    log_path = Path(tempfile.gettempdir()) / "cloud-lab-app.log"
    log = open(log_path, "w", encoding="utf-8")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    STATE.update(proc=proc, port=port, sid=sid, log=log)
    t0 = time.time()
    while time.time() - t0 < boot_timeout:
        st, body = http_get("/healthz", port, timeout=0.5)
        # 只认「我们自己的 app」：healthz 必须返回 {"status": ...}。
        # 这样即使端口被代理/其他服务占用，也不会误判启动成功。
        if st is not None and '"status"' in body:
            print(f"app 已启动 → 127.0.0.1:{port}  scenario={sid or 'healthy'}  "
                  f"tick_period={tick_period}s")
            print(f"进程日志：{log_path}（排障时可读它）")
            return port
        if proc.poll() is not None:
            break
        time.sleep(0.2)
    stop_app()
    print("启动失败，最后 20 行日志：")
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-20:]))
    raise RuntimeError("app 启动失败：端口可能被占用，换一个 port（不传则自动选空闲端口）")


def parse_metrics(text):
    out = {}
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        parts = line.rsplit(" ", 1)
        if len(parts) == 2:
            try:
                out[parts[0]] = float(parts[1])
            except ValueError:
                pass
    return out


def metrics(port=None):
    st, body = http_get("/metrics", port)
    if st != 200:
        raise RuntimeError(f"/metrics 读取失败: {st} {body[:200]}")
    return parse_metrics(body)


def _fmt(v):
    if v is None:
        return "-"
    if abs(v) >= 1000:
        return f"{v:,.0f}"
    if abs(v) >= 10:
        return f"{v:.1f}"
    return f"{v:.2f}"


def show_metrics(m, keys=None):
    keys = keys or BASELINE_KEYS
    rows = "".join(
        f"<tr><td>{LABELS.get(k, k)}</td><td>{_fmt(m.get(k))}</td>"
        f"<td>{REF.get(k, '-')}</td></tr>"
        for k in keys
    )
    display(HTML("<table class='lab-table'><tr><th>指标</th><th>当前值</th>"
                 "<th>健康参考</th></tr>" + rows + "</table>"))


def wait_until(predicate, timeout=90, interval=0.5, label="故障特征"):
    t0 = time.time()
    last_mark = -1
    while time.time() - t0 < timeout:
        m = metrics()
        if predicate(m):
            print(f"✅ 已捕获「{label}」，用时 {time.time() - t0:.1f}s")
            return m
        elapsed = int(time.time() - t0)
        if elapsed and elapsed % 5 == 0 and elapsed != last_mark:
            last_mark = elapsed
            print(f"   ...等待 {label} ({elapsed}s)  cpu={m.get('node_cpu_percent', 0):.0f}%  "
                  f"p99={m.get('http_request_duration_ms_p99', 0):.0f}ms  "
                  f"pool={m.get('app_conn_pool_in_use', 0):.0f}/"
                  f"{m.get('app_conn_pool_size', 0):.0f}  "
                  f"db={m.get('db_cpu_percent', 0):.0f}%")
        time.sleep(interval)
    raise TimeoutError(f"等待「{label}」超时（{timeout}s）：检查 scenario 名与 tick_period")


def watch(seconds=8, interval=1.0, keys=None):
    keys = keys or BASELINE_KEYS
    t0 = time.time()
    rows = []
    while time.time() - t0 < seconds:
        rows.append((time.time() - t0, metrics()))
        time.sleep(interval)
    head = "".join(f"<th>{LABELS.get(k, k)}</th>" for k in keys)
    body = "".join(
        "<tr><td>{:.1f}s</td>{}</tr>".format(
            t, "".join(f"<td>{_fmt(m.get(k))}</td>" for k in keys))
        for t, m in rows
    )
    display(HTML("<table class='lab-table'><tr><th>t</th>" + head + "</tr>" + body + "</table>"))


def sample_api(n=20):
    codes = [http_get("/api/order/list")[0] for _ in range(n)]
    c = Counter(codes)
    bad = sum(v for k, v in c.items() if k and int(k) >= 500)
    print(f"直连 /api/order/list {n} 次：{dict(c)}    5xx 占比={bad / n:.0%}")
    return c


def run_evidence(sid, keys):
    cp = subprocess.run([sys.executable, str(LAB / "sim" / "runner.py"),
                         "--scenario", sid, "--out", str(EVID)],
                        capture_output=True, text=True)
    if cp.returncode != 0:
        print(cp.stdout)
        print(cp.stderr)
        raise RuntimeError("runner 执行失败")
    s = json.loads((EVID / sid / "summary.json").read_text(encoding="utf-8"))
    pre, fault = s["phases"]["pre"], s["phases"]["fault"]

    def stat(k):
        v = fault.get(k)
        if v is None:
            return "-"
        if k == "http_5xx_rate":
            return f"{v:.1%}"
        return _fmt(v)

    def delta(k):
        a, b = pre.get(k), fault.get(k)
        if a is None or b is None:
            return "-"
        if k.endswith("__delta"):
            return "故障期增量"
        if k == "http_5xx_rate":
            return f"{(b - a) * 100:+.1f} pp"
        if k in ("cache_hit_ratio",):
            return f"{b - a:+.3f}"
        if not a:
            return "—"
        return f"{(b - a) / a * 100:+.0f}%"

    rows = "".join(
        f"<tr><td>{LABELS.get(k, k)}</td><td>{_fmt(pre.get(k))}</td>"
        f"<td>{stat(k)}</td><td>{delta(k)}</td></tr>" for k in keys
    )
    display(HTML(
        f"<div class='lab-note'><b>{sid}</b> — {s['scenario']['name']}<br>"
        f"阶段：pre [0,{s['run']['phase_boundaries']['pre'][1]}) → "
        f"fault [{s['run']['phase_boundaries']['fault'][0]},"
        f"{s['run']['phase_boundaries']['fault'][1]}) → post；"
        f"墙钟耗时 {s['run']['wall_seconds']}s</div>"
        "<table class='lab-table'><tr><th>指标</th><th>pre（正常）</th>"
        "<th>fault（故障）</th><th>变化</th></tr>" + rows + "</table>"))
    if s["anomalies"]:
        print("实测异常信号：")
        for a in s["anomalies"]:
            print("  -", a)
    if s.get("log_signal_counts"):
        print("日志信号计数：", s["log_signal_counts"])
    print(f"取证文件：{EVID / sid}/summary.json、timeline.md、app.log、metrics.csv")
    return s


def show_logs(sid, keywords, limit=8):
    path = EVID / sid / "app.log"
    if not path.exists():
        print("日志不存在，请先运行 run_evidence()")
        return
    lines = [ln for ln in path.read_text(errors="replace").splitlines()
             if any(k.lower() in ln.lower() for k in keywords)]
    if not lines:
        print("未匹配到日志关键词：", keywords)
        return
    print(f"日志命中 {len(lines)} 行，摘录（首 2 + 末 4）：")
    picked = lines[:2] + (["..."] if len(lines) > 6 else []) + lines[-4:]
    for ln in picked:
        print("  ", ln[:220])


def verify_recovery(keys=None, wait_s=2.0):
    keys = keys or BASELINE_KEYS
    start_app(None, tick_period=0.1)
    try:
        time.sleep(wait_s)
        m = metrics()
        show_metrics(m, keys)
        hz = http_get("/healthz")[0]
        codes = [http_get("/api/order/list")[0] for _ in range(10)]
        ok = hz == 200 and all(c == 200 for c in codes)
        print(("✅ 已恢复到健康基线" if ok else "❌ 仍未恢复")
              + f"：healthz={hz}  api={codes}")
    finally:
        stop_app()


print("工具箱就绪。常用：start_app / stop_app / metrics / wait_until / watch / "
      "run_evidence / show_logs / verify_recovery")


## Lab 0｜先认识「正常」：基线观察

**难度** ★☆☆ ｜ **能力** 建立基线感、读指标

> 排障最怕的不是没数据，而是不知道「正常长什么样」。先跑一次健康态，记住三层的关键值。

### 1. 原理

健康态下：没有 5xx，P99 约 200ms（因为 15% 请求回源 DB，单次约 150ms），
节点 CPU 25%、内存 42%、磁盘 55%、inode 30%，连接池约 6/10，缓存命中率约 0.85。

### 2. 跑一次健康基线


In [ ]:
try:
    start_app(None, tick_period=0.1)
    time.sleep(3)
    print("当前快照：")
    show_metrics(metrics(), BASELINE_KEYS)
    print("\n连续观察 10 秒（每 1 秒采样）：")
    watch(seconds=10, keys=BASELINE_KEYS)
    print("\nhealthz =", http_get("/healthz")[0])
    print("直连接口 10 次 =", [http_get("/api/order/list")[0] for _ in range(10)])
finally:
    stop_app()


### 3. 观察任务（先自己回答）

1. 三层里，哪三个指标最能代表「应用层健康」？哪三个代表「数据层健康」？
2. 为什么健康态的 P99 约 200ms，而不是个位数？这说明了什么？
3. 如果客户说「你们的接口很慢」，你会先看 P50 还是 P99？为什么？

### 4. 参考结论

- **应用层**：`http_requests_total{status="5xx"}`、P95/P99、`app_worker_busy`。
- **数据层**：`db_cpu_percent`、`db_slow_queries_total`、`app_conn_pool_in_use`、`cache_latency_ms`。
- P99 约 200ms 是**基线**：缓存未命中的请求要回源 MySQL（约 150ms）。所以「P99 升高」要相对于
  基线判断，而不是看到 200ms 就喊慢。
- 客户说慢，先看 **P99/P95**（尾部体验），再看 P50（整体），最后按接口/依赖拆分。

> 记住这四个「基线锚点」：**5xx=0、P99≈200ms、CPU≈25%、池≈6/10**。后面 8 个 Lab 都是它们被打破。



## Lab 1｜接入层：SLB 健康检查失败，网关 502 而后端健康

**难度** ★★☆ ｜ **能力标签** SLB / 主动健康检查 / 分层定位

### 1. 原理（先懂再动手）

云 SLB/CLB/ELB 用**主动健康检查**：每隔 N 秒主动请求后端 `/healthz`，连续失败若干次就把实例摘除。摘除后客户端拿到 502/504，但请求根本没到应用，所以**应用自己的 5xx 是 0**。这正是「二阶问题」：后端资源正常、业务接口也正常，只有接入层判它不健康。

> **类比**：分诊台护士每 2 秒摸一次脉搏；摸不到就把病人推出门外——里面医生再好也没用。

**本实验注入**：`healthcheck_fail=true` → `/healthz` 返回 500；`lb_drop_ratio=0.5` → 一半请求在 SLB 被丢弃。


### 2. 跑完整时间线（确定性证据，约 8 秒）

先用 runner 把 pre → fault → post 完整跑一遍，得到可复盘的阶段对比。

In [ ]:
s = run_evidence("S5-lb-healthcheck-5xx", ['gateway_upstream_5xx_total__delta', 'req_rate_per_tick', 'http_5xx_rate', 'node_cpu_percent', 'http_request_duration_ms_p99'])
show_logs("S5-lb-healthcheck-5xx", ['health check failed', 'upstream timed out', 'no live upstreams'])


In [ ]:
try:
    start_app("S5-lb-healthcheck-5xx", tick_period=0.1)
    wait_until(lambda m: http_get("/healthz")[0] == 500, label="healthz 变 500（后端被判不健康）", timeout=90)
    print("\n直连接口抽样：")
    sample_api(20)
    print("\n实时指标（每 1s 采样，窗口 8s）：")
    watch(seconds=8, keys=['gateway_upstream_5xx_total', 'http_requests_total{status="2xx"}', 'http_requests_total{status="5xx"}', 'http_request_duration_ms_p99', 'node_cpu_percent'])
    print("\nhealthz =", http_get("/healthz")[0])
finally:
    stop_app()


### 3. 观察任务（先自己回答，再看下一节）

1. 为什么应用 5xx 是 0，网关 5xx 却大幅增长？
2. 为什么业务接口到达率从约 118 掉到约 60 req/tick？
3. 如果只看应用 CPU/内存，会得出什么错误结论？

> 提示：先看 RED（错误率、P99、到达量），再看 USE（CPU、池、磁盘、内存），最后用日志确认。不要一上来就重启。

### 4. 参考结论

1. **证据链**：`gateway_upstream_5xx_total` 故障期新增约 7000 次；到达率 118→60 req/tick；应用 `http_5xx_rate=0`、P99 约 195ms 正常、CPU 25%。**错误在接入层，不在后端。**
2. **根因**：探针 `/healthz` 返回 500 → SLB 连续失败后摘除后端 → 剩余流量被丢弃/返回 502。`lb_drop_ratio=0.5` 让一半请求根本没到达应用，所以应用侧看不到错误。
3. **止损**：先把探针改成 TCP/端口探测或临时放宽 `fall/rise`；如果是探针路径配错，回滚配置；多实例时先摘除异常实例，保住其余实例。
4. **根治**：让探针反映真实业务健康（检查 DB/缓存依赖与超时）、统一探针路径与阈值、给「健康检查失败」单独配告警，避免和业务 5xx 混在一起。

**定位动作（值班时按这个顺序）**

1. 看网关 5xx 与请求到达量：到达量下降 + 应用 5xx≈0 → 高度怀疑后端被摘。
2. 直连后端 `/healthz` 与 `/api/order/list`：healthz 500、业务 200 → 探针与业务健康不一致。
3. 核对 SLB 探针配置：路径、期望状态码、超时、`fall/rise`。
4. 看 SLB/ingress 日志：`health check failed`、`no live upstreams`、`502`。

**止损（先恢复业务）**

- 临时把健康检查改为 TCP 探测，或把 `fall` 调大，先恢复流量。
- 回滚最近一次探针/网关配置变更。
- 多实例场景下摘除异常实例，避免整组被摘。

**根治（防止复发）**

- 修正 `/healthz` 实现：只检查关键依赖，设置短超时，失败要能快速返回。
- 统一 SLB 探针路径/状态码/超时与业务真实健康语义。
- 为「健康检查失败率」和「后端可用实例数」建告警，而不是只看业务 5xx。

**验证标准**：healthz 恢复 200，网关 5xx 停止增长，请求到达率回到约 118 req/tick。

**常见误判**

- 看到应用 5xx=0 就下结论「应用没问题」——应用没问题 ≠ 链路没问题。
- 把 502 当成应用报错去查代码，忽略请求可能根本没到应用。

### 5. 恢复验证（本实验的「修好了」长什么样）

重启为健康态，确认健康检查与业务接口恢复，关键指标回到基线。

In [ ]:
verify_recovery(['node_cpu_percent', 'http_request_duration_ms_p99', 'http_requests_total{status="5xx"}'])


### 6. 面试一句话与追问

> 「接入层 5xx 先确认请求有没有到达后端：后端 5xx 为 0、到达量下降、资源正常，基本可判定是健康检查摘除；再查探针路径、期望状态码和 fall/rise 阈值。」

**可能的追问**：如果客户问「为什么刚才还是好的，突然就挂了？」，你会怎么回答？
（参考结构：说明触发条件 → 说明影响面 → 说明已采取的止损 → 给出后续改进，不甩锅、不承诺未验证的结论。）


## Lab 2｜网络层：DNS 解析异常，30% 请求间歇性 5xx

**难度** ★★☆ ｜ **能力标签** DNS / CoreDNS / 间歇性故障

### 1. 原理（先懂再动手）

应用每次访问下游域名都要解析。DNS 上游抖动或 `resolv.conf`/CoreDNS 配置不当，会让一部分请求在解析阶段超时。典型特征是**间歇性**（不是全挂）、**P99 卡在解析超时量级**、而 CPU/内存/DB 全部正常。

> **类比**：电话簿偶尔缺页：大部分电话能打通，但每 10 个里有 3 个查不到号码。

**本实验注入**：`dns_fail_rate=0.30`（30% 请求解析失败）+ `dns_timeout_ms=340`（每次失败耗时约 340ms）。


### 2. 跑完整时间线（确定性证据，约 8 秒）

先用 runner 把 pre → fault → post 完整跑一遍，得到可复盘的阶段对比。

In [ ]:
s = run_evidence("S7-dns-resolution-fail", ['http_5xx_rate', 'http_request_duration_ms_p99', 'http_request_duration_ms_p95', 'node_cpu_percent', 'gateway_upstream_5xx_total__delta'])
show_logs("S7-dns-resolution-fail", ['nameserver', 'getaddrinfo'])


In [ ]:
try:
    start_app("S7-dns-resolution-fail", tick_period=0.1)
    wait_until(lambda m: m.get("http_requests_total{status=\"5xx\"}", 0) > 100, label="5xx 累计开始上升", timeout=90)
    print("\n直连接口抽样：")
    sample_api(20)
    print("\n实时指标（每 1s 采样，窗口 8s）：")
    watch(seconds=8, keys=['http_requests_total{status="5xx"}', 'http_request_duration_ms_p99', 'http_request_duration_ms_p95', 'node_cpu_percent', 'gateway_upstream_5xx_total'])
    print("\nhealthz =", http_get("/healthz")[0])
finally:
    stop_app()


### 3. 观察任务（先自己回答，再看下一节）

1. 为什么错误率约 30%，而不是 0% 或 100%？
2. 为什么 P99 约 435ms，而不是秒级或正常 200ms？
3. 怎么证明问题不在应用代码或数据库？

> 提示：先看 RED（错误率、P99、到达量），再看 USE（CPU、池、磁盘、内存），最后用日志确认。不要一上来就重启。

### 4. 参考结论

1. **证据链**：5xx 约 30.1%；P99 199→435ms；CPU 25%、DB CPU 22%、连接池正常；日志出现 `nameserver ... timeout` / `getaddrinfo failed`。
2. **根因**：30% 的请求在 DNS 解析阶段失败；失败耗时约等于解析超时（340ms），所以 P99 被抬到「基线 + 超时」量级，而不是 DB 慢那种秒级。
3. **止损**：切换备用 DNS/改 hosts、临时把下游地址改成 IP、给解析加短超时和重试；如果是 CoreDNS 问题，重启/扩容 CoreDNS。
4. **根治**：修 `resolv.conf` 与 `ndots`、减少无谓 search domain、给 DNS 配监控与缓存、应用侧加解析缓存和熔断。

**定位动作（值班时按这个顺序）**

1. 看错误是否间歇、是否集中在某几个下游域名。
2. 对比 P99 与解析超时量级：卡在 300~500ms 高度怀疑 DNS。
3. 在容器内 `nslookup/getent hosts 下游域名`，看 `/etc/resolv.conf` 与 `ndots`。
4. 查 CoreDNS 日志/上游 DNS 可用性，确认是否只有集群内解析异常。

**止损（先恢复业务）**

- 切换备用 DNS 或临时写 hosts 指向可用 IP。
- 给解析增加短超时 + 有限重试，避免线程被长时间占住。
- CoreDNS 异常时重启/扩容，并观察解析成功率。

**根治（防止复发）**

- 修 `resolv.conf`，合理设置 `ndots` 与 search domain。
- 为 DNS 解析成功率/延迟建监控和告警。
- 应用侧加 DNS 缓存，降低解析放大。

**验证标准**：5xx 回落到 0，P99 回到约 200ms，`nameserver timeout` 日志消失。

**常见误判**

- 把间歇性 5xx 当成应用 bug，反复重启应用。
- 忽略 P99 与超时量级的对应关系，直接怀疑数据库。

### 5. 恢复验证（本实验的「修好了」长什么样）

重启为健康态，确认健康检查与业务接口恢复，关键指标回到基线。

In [ ]:
verify_recovery(['http_request_duration_ms_p99', 'http_requests_total{status="5xx"}', 'node_cpu_percent'])


### 6. 面试一句话与追问

> 「间歇性 5xx + 资源全正常 + P99 卡在几百毫秒，我会先怀疑 DNS：查 resolv.conf、ndots、CoreDNS 和上游解析，再决定切 DNS 还是加解析缓存。」

**可能的追问**：如果客户问「为什么刚才还是好的，突然就挂了？」，你会怎么回答？
（参考结构：说明触发条件 → 说明影响面 → 说明已采取的止损 → 给出后续改进，不甩锅、不承诺未验证的结论。）


## Lab 3｜应用层：数据库连接池耗尽，请求排队超时

**难度** ★★☆ ｜ **能力标签** 连接池 / 连接泄漏 / HikariCP

### 1. 原理（先懂再动手）

应用用连接池复用 DB 连接。池上限 10，但代码泄漏了 10 个连接（借出不还），新请求只能排队等 `conn_timeout`，等不到就 500。关键特征：**池占用 100%，但 DB CPU 正常**——因为请求卡在应用侧排队，根本没打到 DB。

> **类比**：餐厅只有 10 个座位，10 个人坐下后一直不走；新客人排队 400ms 就被请出去。

**本实验注入**：`leaked_conns=10`（等于池上限）+ `cache_hit_ratio=0.80`（更多请求需要真实 DB 连接）。


### 2. 跑完整时间线（确定性证据，约 8 秒）

先用 runner 把 pre → fault → post 完整跑一遍，得到可复盘的阶段对比。

In [ ]:
s = run_evidence("S2-conn-pool-exhaustion", ['app_conn_pool_in_use', 'app_conn_pool_size', 'http_request_duration_ms_p99', 'http_5xx_rate', 'db_cpu_percent'])
show_logs("S2-conn-pool-exhaustion", ['HikariPool', 'connection leak'])


In [ ]:
try:
    start_app("S2-conn-pool-exhaustion", tick_period=0.1)
    wait_until(lambda m: m.get("app_conn_pool_in_use", 0) >= m.get("app_conn_pool_size", 1), label="连接池被打满", timeout=90)
    print("\n直连接口抽样：")
    sample_api(20)
    print("\n实时指标（每 1s 采样，窗口 8s）：")
    watch(seconds=8, keys=['app_conn_pool_in_use', 'app_conn_pool_size', 'http_request_duration_ms_p99', 'http_requests_total{status="5xx"}', 'db_cpu_percent'])
    print("\nhealthz =", http_get("/healthz")[0])
finally:
    stop_app()


### 3. 观察任务（先自己回答，再看下一节）

1. 为什么 DB CPU 只有 22%，应用却大量 5xx？
2. 为什么 P99 恰好约 420ms？这个数字和哪个配置对应？
3. 健康时池也占 6/10，为什么那时不报错？

> 提示：先看 RED（错误率、P99、到达量），再看 USE（CPU、池、磁盘、内存），最后用日志确认。不要一上来就重启。

### 4. 参考结论

1. **证据链**：`app_conn_pool_in_use=10/10`；P99 199→422ms（接近 `conn_timeout_ms=400`）；5xx 约 20.5%；`db_cpu_percent=22%` 正常；缓存命中率 0.85→0.79。
2. **根因**：连接泄漏占满池；未命中缓存的请求需要 DB 连接，等 400ms 后抛「Connection is not available」。DB 侧连接数没打满（应用池只有 10）。
3. **止损**：重启实例释放连接；临时调大池（只治标）；kill 长事务；把泄漏接口降级/限流。
4. **根治**：修连接泄漏（try-with-resources / finally close）、加泄漏检测与最大生命周期、给池占用率配告警，压测验证池容量。

**定位动作（值班时按这个顺序）**

1. 看 `app_conn_pool_in_use / _size` 是否长期 100%。
2. 看 P99 是否等于连接等待超时（本案 400ms）。
3. 看 `db_cpu_percent` 与 DB 服务端连接数：正常 → 卡在应用池，不是 DB 慢。
4. 查日志 `HikariPool ... Connection is not available` 与 `connection leak detected`。

**止损（先恢复业务）**

- 重启异常实例，快速释放泄漏连接。
- 临时调大池上限并限流，给修复争取时间（注意 DB 上限）。
- 定位并 kill 长事务/泄漏请求。

**根治（防止复发）**

- 修代码：连接必须在 finally/try-with-resources 中归还。
- 配置 `leakDetectionThreshold`、`maxLifetime`、合理的 `connectionTimeout`。
- 按「池占用率 + 等待队列」建告警，定期做容量评估。

**验证标准**：池占用回落到约 6/10，P99 回到约 200ms，5xx 归零。

**常见误判**

- 盲目扩池：泄漏存在时只会把问题推迟，还会压垮 DB。
- 只看 DB 连接数：应用池满和 DB 连接满不是一回事。

### 5. 恢复验证（本实验的「修好了」长什么样）

重启为健康态，确认健康检查与业务接口恢复，关键指标回到基线。

In [ ]:
verify_recovery(['app_conn_pool_in_use', 'http_request_duration_ms_p99', 'http_requests_total{status="5xx"}'])


### 6. 面试一句话与追问

> 「池满但 DB CPU 正常、P99 恰好等于连接等待超时，基本就是连接泄漏或长事务占池；先重启止血，再修归还逻辑并加泄漏检测。」

**可能的追问**：如果客户问「为什么刚才还是好的，突然就挂了？」，你会怎么回答？
（参考结构：说明触发条件 → 说明影响面 → 说明已采取的止损 → 给出后续改进，不甩锅、不承诺未验证的结论。）


## Lab 4｜数据层：慢 SQL / 缺失索引，DB CPU 逼近 100%

**难度** ★★★ ｜ **能力标签** MySQL / 执行计划 / 索引 / 慢查询

### 1. 原理（先懂再动手）

新上线的查询走了缺失索引的 SQL，全表扫描把 DB CPU 打到 95%+，慢查询堆积并占用连接，应用侧 P99 进入秒级。它和连接池耗尽的表面现象很像（都池满、都 5xx），**区分点是 DB CPU 和慢查询增量**。

> **类比**：查字典没有目录，每查一个字都从第一页翻到最后——不是手慢，是方法错。

**本实验注入**：`db_cpu_target=96%`、`db_slow_query_rate=0.15`、`db_slow_query_ms=1200`。


### 2. 跑完整时间线（确定性证据，约 8 秒）

先用 runner 把 pre → fault → post 完整跑一遍，得到可复盘的阶段对比。

In [ ]:
s = run_evidence("S3-slow-sql-db-cpu", ['db_cpu_percent', 'db_slow_queries_total__delta', 'http_request_duration_ms_p99', 'http_5xx_rate', 'app_conn_pool_in_use'])
show_logs("S3-slow-sql-db-cpu", ['slow query', 'mysqld CPU'])


In [ ]:
try:
    start_app("S3-slow-sql-db-cpu", tick_period=0.1)
    wait_until(lambda m: m.get("db_cpu_percent", 0) > 80, label="DB CPU 超过 80%", timeout=90)
    print("\n直连接口抽样：")
    sample_api(20)
    print("\n实时指标（每 1s 采样，窗口 8s）：")
    watch(seconds=8, keys=['db_cpu_percent', 'db_slow_queries_total', 'http_request_duration_ms_p99', 'http_request_duration_ms_p95', 'app_conn_pool_in_use'])
    print("\nhealthz =", http_get("/healthz")[0])
finally:
    stop_app()


### 3. 观察任务（先自己回答，再看下一节）

1. 同样是池满 + P99 升高，怎么和 Lab 3 的连接池耗尽区分？
2. 为什么连接池也被占满？这是根因还是结果？
3. 慢查询日志里应该找哪三个信息？

> 提示：先看 RED（错误率、P99、到达量），再看 USE（CPU、池、磁盘、内存），最后用日志确认。不要一上来就重启。

### 4. 参考结论

1. **证据链**：`db_cpu_percent` 22→95.4%；`db_slow_queries_total` 故障期 +364；P99 199→2028ms（秒级）；5xx 约 15.7%；池 10/10。
2. **根因**：缺失索引导致全表扫描 → DB CPU 饱和 → 查询变慢 → 长时间占用连接 → 池被占满。**池满是结果，不是根因。**
3. **止损**：kill 慢查询、对问题接口限流/降级、回滚新上线 SQL；必要时加只读副本分流。
4. **根治**：`EXPLAIN` 看执行计划，补索引/改写 SQL；建慢查询与 DB CPU 告警；上线前做 SQL 审核。

**定位动作（值班时按这个顺序）**

1. 先看 `db_cpu_percent` 与 `db_slow_queries_total`：双高 → 数据层。
2. 对比 Lab 3：连接池耗尽时 DB CPU 正常；慢 SQL 时 DB CPU 接近 100%。
3. 抓慢查询：SQL 文本、`rows_examined`、耗时。
4. `EXPLAIN` 看是否全表扫描、是否命中索引。

**止损（先恢复业务）**

- kill 长事务/慢查询，释放连接。
- 对问题接口限流或降级，避免继续压 DB。
- 回滚最近上线的 SQL/接口变更。

**根治（防止复发）**

- 按查询条件补联合索引，避免函数/隐式转换导致索引失效。
- 改写 SQL，减少回表与全表扫描。
- 上线前 SQL 审核 + 慢查询告警 + 容量评估。

**验证标准**：DB CPU 回落到约 22%，P99 回到约 200ms，慢查询不再增长。

**常见误判**

- 只扩连接池：DB 更忙，问题更糟。
- 不看执行计划就加索引，可能加错或重复索引。

### 5. 恢复验证（本实验的「修好了」长什么样）

重启为健康态，确认健康检查与业务接口恢复，关键指标回到基线。

In [ ]:
verify_recovery(['db_cpu_percent', 'http_request_duration_ms_p99', 'app_conn_pool_in_use'])


### 6. 面试一句话与追问

> 「P99 秒级 + DB CPU 95% + 慢查询增长，先 kill/限流止血，再用 EXPLAIN 定位缺失索引；连接池满只是结果，不要本末倒置。」

**可能的追问**：如果客户问「为什么刚才还是好的，突然就挂了？」，你会怎么回答？
（参考结构：说明触发条件 → 说明影响面 → 说明已采取的止损 → 给出后续改进，不甩锅、不承诺未验证的结论。）


## Lab 5｜应用层：CPU 打满，线程池饱和引发连锁超时

**难度** ★★★ ｜ **能力标签** CPU / 线程池 / 排队 / 连锁假象

### 1. 原理（先懂再动手）

实例 CPU 被死循环/批量任务打到 98%，每次请求的服务时间被放大，工作线程被占满，排队导致超时。连锁反应是：线程持有 DB 连接不放 → 连接池也满，容易误判成 DB 问题。

> **类比**：收银员只剩一个人还特别慢，队伍排到门外；不是货架（DB）没货，是收银台堵了。

**本实验注入**：`node_cpu_target=98%` + `cpu_service_mult=3.4`（服务时间放大 3.4 倍）。


### 2. 跑完整时间线（确定性证据，约 8 秒）

先用 runner 把 pre → fault → post 完整跑一遍，得到可复盘的阶段对比。

In [ ]:
s = run_evidence("S1-app-cpu-saturation", ['node_cpu_percent', 'http_request_duration_ms_p95', 'http_request_duration_ms_p99', 'http_5xx_rate', 'app_worker_busy'])
show_logs("S1-app-cpu-saturation", ['worker pool', 'request timeout', 'cpu='])


In [ ]:
try:
    start_app("S1-app-cpu-saturation", tick_period=0.1)
    wait_until(lambda m: m.get("node_cpu_percent", 0) > 80, label="节点 CPU 超过 80%", timeout=90)
    print("\n直连接口抽样：")
    sample_api(20)
    print("\n实时指标（每 1s 采样，窗口 8s）：")
    watch(seconds=8, keys=['node_cpu_percent', 'app_worker_busy', 'http_request_duration_ms_p99', 'http_requests_total{status="5xx"}', 'app_conn_pool_in_use'])
    print("\nhealthz =", http_get("/healthz")[0])
finally:
    stop_app()


### 3. 观察任务（先自己回答，再看下一节）

1. 为什么连接池也满了？这是 DB 问题吗？
2. P99 约 951ms，和连接池耗尽的 422ms、慢 SQL 的 2028ms 有什么形态差异？
3. CPU 高就一定是流量涨了吗？还可能是什么？

> 提示：先看 RED（错误率、P99、到达量），再看 USE（CPU、池、磁盘、内存），最后用日志确认。不要一上来就重启。

### 4. 参考结论

1. **证据链**：CPU 25→97.4%；P95 179→814ms、P99 199→951ms；5xx 约 13.2%；`app_worker_busy` 升到 17（池 16）；连接池 10/10；DB CPU 22% 正常。
2. **根因**：CPU 饱和 → 服务时间放大 → 工作线程打满 → 请求排队超时；线程长时间持有连接，导致池满的**假象**。
3. **止损**：扩容/摘除异常实例、限流、回滚可疑发布、把批量任务挪走。
4. **根治**：`top -H` + 线程栈/火焰图定位热点；加 HPA/容量水位；批任务错峰；给 CPU 与线程池排队建告警。

**定位动作（值班时按这个顺序）**

1. 先看 CPU 与 `app_worker_busy`：双高且 P99 升高 → 应用资源饱和。
2. 对比 DB CPU：正常 → 不是数据库慢。
3. `top -H -p <pid>` 找高 CPU 线程，再抓线程栈/火焰图。
4. 核对最近发布、批任务、流量突增与实例数。

**止损（先恢复业务）**

- 扩容或重启异常实例，先把流量接住。
- 限流/降级非核心接口，给核心链路留 CPU。
- 暂停批量任务或错峰执行。

**根治（防止复发）**

- 定位死循环/低效代码并优化，必要时异步化。
- 配置 HPA 与 CPU 水位告警，做容量评估。
- 批任务隔离到独立实例/队列，避免抢占在线流量。

**验证标准**：CPU 回落到约 25%，忙碌线程回落到约 8/16，P99 回到约 200ms。

**常见误判**

- 看到池满就查 DB，忽略 CPU 这个上游根因。
- 直接重启不保留现场，热点线程证据丢失。

### 5. 恢复验证（本实验的「修好了」长什么样）

重启为健康态，确认健康检查与业务接口恢复，关键指标回到基线。

In [ ]:
verify_recovery(['node_cpu_percent', 'app_worker_busy', 'http_request_duration_ms_p99'])


### 6. 面试一句话与追问

> 「CPU 饱和会制造连接池满的假象：先看 DB CPU 是否正常，再看 top -H 和线程栈；止损靠扩容/限流，根治靠定位热点代码和容量建设。」

**可能的追问**：如果客户问「为什么刚才还是好的，突然就挂了？」，你会怎么回答？
（参考结构：说明触发条件 → 说明影响面 → 说明已采取的止损 → 给出后续改进，不甩锅、不承诺未验证的结论。）


## Lab 6｜缓存层：Redis 热 key，延迟抖动与回源放大

**难度** ★★★ ｜ **能力标签** Redis / 热 key / 大 key / 缓存命中率

### 1. 原理（先懂再动手）

某个热 key 被高频访问，单分片 CPU/网络被打满，缓存 P95 从 4ms 涨到几百毫秒，部分请求超时；同时缓存未命中增加，回源把 DB CPU 也抬高。**缓存问题会伪装成 DB 问题**，要看 `cache_latency_ms` 和命中率。

> **类比**：全城人都打同一个咨询电话，线路占满；其他电话也打不进去。

**本实验注入**：`cache_hotkey=true`、`cache_hotkey_ms=120`、`cache_hotkey_prob=0.5`、`cache_timeout_rate=0.06`、命中率降到 0.75。


### 2. 跑完整时间线（确定性证据，约 8 秒）

先用 runner 把 pre → fault → post 完整跑一遍，得到可复盘的阶段对比。

In [ ]:
s = run_evidence("S8-redis-hotkey", ['cache_latency_ms', 'cache_hit_ratio', 'http_request_duration_ms_p99', 'http_5xx_rate', 'db_cpu_percent'])
show_logs("S8-redis-hotkey", ['Redis command timed out', 'hot key'])


In [ ]:
try:
    start_app("S8-redis-hotkey", tick_period=0.1)
    wait_until(lambda m: m.get("cache_latency_ms", 0) > 100, label="缓存 P95 超过 100ms", timeout=90)
    print("\n直连接口抽样：")
    sample_api(20)
    print("\n实时指标（每 1s 采样，窗口 8s）：")
    watch(seconds=8, keys=['cache_latency_ms', 'cache_hit_ratio', 'http_request_duration_ms_p99', 'db_cpu_percent', 'http_requests_total{status="5xx"}'])
    print("\nhealthz =", http_get("/healthz")[0])
finally:
    stop_app()


### 3. 观察任务（先自己回答，再看下一节）

1. 为什么 DB CPU 也从 22% 升到约 45%？这是 DB 的锅吗？
2. 命中率只从 0.85 降到 0.75，为什么影响这么大？
3. 怎么在 Redis 侧确认是热 key 而不是普通慢查询？

> 提示：先看 RED（错误率、P99、到达量），再看 USE（CPU、池、磁盘、内存），最后用日志确认。不要一上来就重启。

### 4. 参考结论

1. **证据链**：`cache_latency_ms` 4.4→369ms；命中率 0.85→0.754；P99 199→629ms；DB CPU 22→44.8%（回源放大）；5xx 约 5.8%；日志 `hot key detected` / `Redis command timed out`。
2. **根因**：热 key 集中在单分片，缓存响应抖动/超时；未命中请求回源，抬高 DB CPU。DB 升高是**结果**。
3. **止损**：本地缓存热 key、对热 key 做打散/多副本、临时扩容分片、对回源加限流。
4. **根治**：`redis-cli --hotkeys`/`--bigkeys`/`SLOWLOG` 定位；拆分大 key、热点 key 加本地缓存、分片均衡与容量水位告警。

**定位动作（值班时按这个顺序）**

1. 先看 `cache_latency_ms` 与 `cache_hit_ratio`，再看 DB CPU。
2. `redis-cli --hotkeys`、`--bigkeys`、`SLOWLOG GET` 定位 key。
3. 看是否单分片/单节点 CPU 偏高（分片倾斜）。
4. 对比「缓存延迟高 + DB 回源升高」与「DB 自身慢查询」：前者慢查询不增长。

**止损（先恢复业务）**

- 热 key 加进程内本地缓存，减少对 Redis 的集中访问。
- 临时扩容/迁移分片，或给热点读走副本。
- 对回源 DB 限流，防止缓存故障拖垮数据库。

**根治（防止复发）**

- 拆分大 key，热点 key 打散/多副本。
- 建缓存延迟、命中率、分片 CPU 告警。
- 做容量评估与压测，避免单分片热点。

**验证标准**：缓存 P95 回到约 4ms，命中率回到约 0.85，DB CPU 回落到约 22%。

**常见误判**

- 只扩容 Redis 不分片，热 key 还是打在同一分片。
- 用 `DEL` 删大 key 阻塞主线程，应该用 `UNLINK` 或分批删。

### 5. 恢复验证（本实验的「修好了」长什么样）

重启为健康态，确认健康检查与业务接口恢复，关键指标回到基线。

In [ ]:
verify_recovery(['cache_latency_ms', 'cache_hit_ratio', 'http_request_duration_ms_p99'])


### 6. 面试一句话与追问

> 「缓存延迟高 + 命中率下降 + DB CPU 被动升高，优先查热 key/大 key；DB 升高是回源结果，别先去优化 SQL。」

**可能的追问**：如果客户问「为什么刚才还是好的，突然就挂了？」，你会怎么回答？
（参考结构：说明触发条件 → 说明影响面 → 说明已采取的止损 → 给出后续改进，不甩锅、不承诺未验证的结论。）


## Lab 7｜基础设施：磁盘 / inode 打满，写入快速失败

**难度** ★★★ ｜ **能力标签** Linux / 磁盘 / inode / logrotate

### 1. 原理（先懂再动手）

日志或临时文件把磁盘/inode 写满，应用写文件失败并返回 5xx。关键形态：**P99 正常，但 5xx 明显上升**——因为写入失败是「快失败」（几毫秒返回），不是「慢超时」。另外要区分**空间满**（`df -h`）和 **inode 满**（`df -i`）。

> **类比**：图书馆还有空地，但所有书架编号都用完了——新书没号可上架。

**本实验注入**：`disk_fill_rate=0.62%/tick`、`inode_fill_rate=0.70%/tick`，故障期逐步逼近 100%。


### 2. 跑完整时间线（确定性证据，约 8 秒）

先用 runner 把 pre → fault → post 完整跑一遍，得到可复盘的阶段对比。

In [ ]:
s = run_evidence("S6-disk-inode-full", ['node_disk_percent__max', 'node_inode_percent__max', 'http_5xx_rate', 'http_request_duration_ms_p99'])
show_logs("S6-disk-inode-full", ['No space left', 'inode exhausted'])


In [ ]:
try:
    start_app("S6-disk-inode-full", tick_period=0.1)
    wait_until(lambda m: m.get("node_disk_percent", 0) >= 99.5 or m.get("node_inode_percent", 0) >= 99.5, label="磁盘或 inode 接近 100%", timeout=90)
    print("\n直连接口抽样：")
    sample_api(20)
    print("\n实时指标（每 1s 采样，窗口 8s）：")
    watch(seconds=8, keys=['node_disk_percent', 'node_inode_percent', 'http_requests_total{status="5xx"}', 'http_request_duration_ms_p99'])
    print("\nhealthz =", http_get("/healthz")[0])
finally:
    stop_app()


### 3. 观察任务（先自己回答，再看下一节）

1. 为什么 P99 几乎没变，却有约 15% 的 5xx？
2. 空间满和 inode 满，分别用什么命令确认？
3. 磁盘满之后，为什么日志可能「看不到错误」？

> 提示：先看 RED（错误率、P99、到达量），再看 USE（CPU、池、磁盘、内存），最后用日志确认。不要一上来就重启。

### 4. 参考结论

1. **证据链**：磁盘峰值 100%、inode 峰值 100%；5xx 约 15.1%；P99 199ms 正常；CPU/内存/DB 全部正常；日志 `No space left on device` / `inode exhausted`。
2. **根因**：磁盘/inode 打满 → 写日志/临时文件失败 → 应用快速返回 500。因为失败很快，延迟分位几乎不变。
3. **止损**：清理最占空间的文件/目录、压缩或转储日志、扩容磁盘、把日志目录迁到独立盘。
4. **根治**：修 `logrotate`（轮转+保留+压缩）、给临时目录做清理策略、监控磁盘与 inode 双指标、设置配额。

**定位动作（值班时按这个顺序）**

1. `df -h` 看空间，`df -i` 看 inode；两个都要看。
2. `du -xh --max-depth=1 /` 找大目录；大量小文件用 `find ... -size`/`lsof` 辅助。
3. 看 5xx 是否集中在写接口，且延迟很短（快失败）。
4. 确认日志能否写入：`touch /var/log/test` 可能直接报 No space left。

**止损（先恢复业务）**

- 清理日志/临时文件，优先恢复写入。
- 临时扩容磁盘或挂载新盘。
- 关闭非核心日志/采样，降低写入量。

**根治（防止复发）**

- 修 logrotate：按大小/时间轮转、压缩、保留 N 份。
- 临时目录定期清理，设置磁盘配额。
- 磁盘空间与 inode 双指标告警，容量水位纳入巡检。

**验证标准**：磁盘与 inode 回落到基线以下，5xx 归零，应用可正常写日志。

**常见误判**

- 只 `df -h` 不看 `df -i`，inode 满时会误判「还有空间」。
- 清空日志但不修 logrotate，很快再次写满。

### 5. 恢复验证（本实验的「修好了」长什么样）

重启为健康态，确认健康检查与业务接口恢复，关键指标回到基线。

In [ ]:
verify_recovery(['node_disk_percent', 'node_inode_percent', 'http_requests_total{status="5xx"}'])


### 6. 面试一句话与追问

> 「5xx 上升但 P99 正常，是典型的快失败；先 `df -h` 和 `df -i` 双查，再定位大文件/小文件，最后修 logrotate 和配额。」

**可能的追问**：如果客户问「为什么刚才还是好的，突然就挂了？」，你会怎么回答？
（参考结构：说明触发条件 → 说明影响面 → 说明已采取的止损 → 给出后续改进，不甩锅、不承诺未验证的结论。）


## Lab 8｜容器生命周期：内存泄漏 → OOMKilled → 反复重启

**难度** ★★★★ ｜ **能力标签** 内存 / OOMKilled / 重启 / 锯齿曲线

### 1. 原理（先懂再动手）

应用内存持续泄漏，RSS 逐步逼近 limit，被内核 OOMKill 后容器重启，内存曲线呈**锯齿**。关键陷阱：**平均内存可能只有 66%，但峰值 99.8%**；平均延迟也正常，只有重启计数在涨。所以要看 `max` 和 `container_restarts_total`，不能只看均值。

> **类比**：行李箱每次出行都多塞一点，直到拉链崩开；重新装箱后又从头开始塞。

**本实验注入**：`mem_leak_rate=1.7%/tick`（约 34 tick 到 100%）+ `oom_enabled=true`。


### 2. 跑完整时间线（确定性证据，约 8 秒）

先用 runner 把 pre → fault → post 完整跑一遍，得到可复盘的阶段对比。

In [ ]:
s = run_evidence("S4-memory-leak-oomkill", ['node_mem_percent__max', 'container_restarts_total__delta', 'http_request_duration_ms_p99', 'http_5xx_rate'])
show_logs("S4-memory-leak-oomkill", ['Out of memory', 'restarting container'])


In [ ]:
try:
    start_app("S4-memory-leak-oomkill", tick_period=0.1)
    wait_until(lambda m: m.get("container_restarts_total", 0) >= 1 or m.get("node_mem_percent", 0) > 99, label="内存逼近 100% 或发生重启", timeout=90)
    print("\n直连接口抽样：")
    sample_api(20)
    print("\n实时指标（每 1s 采样，窗口 8s）：")
    watch(seconds=8, keys=['node_mem_percent', 'container_restarts_total', 'http_request_duration_ms_p99', 'http_requests_total{status="5xx"}'])
    print("\nhealthz =", http_get("/healthz")[0])
finally:
    stop_app()


### 3. 观察任务（先自己回答，再看下一节）

1. 平均内存只有 66%，为什么会 OOM？应该看均值还是峰值？
2. 为什么 5xx 只有约 0.25%，P99 也正常？
3. 怎么区分「内存泄漏」和「流量增长导致内存上涨」？

> 提示：先看 RED（错误率、P99、到达量），再看 USE（CPU、池、磁盘、内存），最后用日志确认。不要一上来就重启。

### 4. 参考结论

1. **证据链**：`node_mem_percent` 峰值 99.8%、均值 66.4%；`container_restarts_total` 故障期 +3；P99 198ms 正常；5xx 约 0.25%；日志 `Out of memory: Killed process` / `OOMKilled`。
2. **根因**：内存泄漏 → 触及 limit → OOMKill → 重启 → 内存回落 → 再泄漏，形成锯齿。重启很快，业务只丢少量请求，所以 5xx 很低，容易被忽略。
3. **止损**：临时抬高内存 limit、滚动重启、降流量/限流；保留现场（heap dump、事件）再重启。
4. **根治**：用 heap/pprof 定位泄漏；修对象生命周期；设置 80% 水位告警；配置合理的 limit/request 与 HPA。

**定位动作（值班时按这个顺序）**

1. 看 `node_mem_percent` 的**峰值**与曲线形态：锯齿 → 泄漏 + 重启。
2. 看 `container_restarts_total` 是否持续增长。
3. `kubectl describe pod` 看 `OOMKilled`/exit 137，`dmesg` 看内核日志。
4. 对比流量曲线：流量平稳但内存持续涨 → 泄漏。

**止损（先恢复业务）**

- 临时抬高 limit 或滚动重启，先稳住业务。
- 限流/降载，减缓泄漏速度。
- 重启前抓 heap dump / 保留事件，避免证据丢失。

**根治（防止复发）**

- 用 heap dump / pprof 定位泄漏对象并修复。
- 内存 80% 水位告警 + 重启次数告警。
- 合理设置 limit/request，做容量评估与压测。

**验证标准**：内存曲线不再锯齿，重启计数停止增长，内存回落到约 42%。

**常见误判**

- 只看平均内存：均值 66% 看起来「还行」，但峰值已经 OOM。
- 只看 5xx：重启太快，5xx 很少，容易漏掉真正的故障。

### 5. 恢复验证（本实验的「修好了」长什么样）

重启为健康态，确认健康检查与业务接口恢复，关键指标回到基线。

In [ ]:
verify_recovery(['node_mem_percent', 'container_restarts_total', 'http_request_duration_ms_p99'])


### 6. 面试一句话与追问

> 「内存要看峰值和重启次数：均值 66% 但峰值 99.8% + 重启计数增长，就是泄漏引发的 OOMKill；先保现场再重启，之后用 heap dump 定位。」

**可能的追问**：如果客户问「为什么刚才还是好的，突然就挂了？」，你会怎么回答？
（参考结构：说明触发条件 → 说明影响面 → 说明已采取的止损 → 给出后续改进，不甩锅、不承诺未验证的结论。）

---

## 鉴别诊断：8 个场景放在一张表里

排障的难点不是「单个场景会不会」，而是「现象相似时能不能分开」。这张表建议背下来。

| 场景 | 最先异常的指标 | 典型形态 | 关键鉴别点 | 最容易混淆 |
|---|---|---|---|---|
| S5 SLB 健康检查 | `gateway_upstream_5xx_total`、到达量 | 网关 5xx 暴增，应用 5xx=0，到达量约减半 | 请求根本没到应用，后端资源全正常 | 应用 5xx 故障 |
| S7 DNS | 5xx 比例 | 约 30% 间歇 5xx，P99 约 400ms | 资源全正常，错误间歇，日志有 nameserver timeout | 应用随机 bug |
| S2 连接池 | `app_conn_pool_in_use` | 池 10/10，P99≈400ms，5xx≈20% | DB CPU 正常，池等待超时 | S3 慢 SQL |
| S3 慢 SQL | `db_cpu_percent`、慢查询 | DB CPU≈95%，P99 秒级，5xx≈15% | 慢查询增长，执行计划全表扫描 | S2 连接池 |
| S1 CPU | `node_cpu_percent`、worker | CPU≈97%，P99≈950ms，线程池满 | DB CPU 正常，CPU/worker 先满 | S2/S3 |
| S8 热 key | `cache_latency_ms` | 缓存 4→369ms，命中率↓，DB CPU≈45% | 缓存延迟远早于 DB 升高 | S3 慢 SQL |
| S6 磁盘/inode | `node_disk/inode_percent` | 双 100%，5xx≈15%，P99 正常 | **快失败**，延迟不变 | 应用写 bug |
| S4 内存泄漏 | `node_mem_percent` 峰值、重启 | 锯齿，峰值 99.8%，重启 +3，5xx 极少 | 看峰值/重启，不看均值 | 流量增长 |

### 一条决策链（从告警到结论）

```text
看到 5xx / 延迟告警
├─ 先看「请求到达量」是否下降
│  ├─ 下降 + 应用 5xx≈0 + 后端资源正常 → S5 接入层健康检查
│  └─ 未下降 → 继续
├─ 看错误是「间歇」还是「持续」
│  ├─ 间歇 ~30% + 资源正常 + P99 卡在几百 ms → S7 DNS
│  └─ 持续 → 继续
├─ 看 DB CPU / 慢查询
│  ├─ DB CPU≈95% + 慢查询增长 → S3 慢 SQL
│  ├─ DB CPU 正常 + 池 100% + P99≈连接超时 → S2 连接池
│  └─ 缓存延迟先飙升 + 命中率下降 → S8 热 key
├─ 看节点资源
│  ├─ CPU≈97% + worker 满 → S1 CPU
│  ├─ 磁盘/inode≈100% + P99 正常 → S6 磁盘/inode
│  └─ 内存锯齿 + 重启增加 → S4 OOM
└─ 都没有明显异常 → 回到接入层/网络/变更记录继续查
```

> 关键原则：**谁先变，谁更可能是根因**。连接池满、DB CPU 升高、网关 5xx 都可能只是结果。


---

## 综合盲测：不看答案，独立值班一次

下面会随机注入一个场景，**不告诉你是哪一个**。只给你告警摘要和接口抽样。
请在纸上写下：

1. 影响面：错误率、延迟、哪些请求受影响？
2. 分层：接入层 / 应用层 / 数据层 / 基础设施，问题在哪一层？
3. 根因假设：最像哪个场景？用什么证据排除最相似的另一个？
4. 止损动作：先说两个立即能做的。
5. 根治动作：再说两个防止复发的。

写完再运行下一格揭晓答案。

In [ ]:
BLIND_NAMES = {
    "S5-lb-healthcheck-5xx": "SLB 健康检查失败",
    "S7-dns-resolution-fail": "DNS 解析异常",
    "S2-conn-pool-exhaustion": "数据库连接池耗尽",
    "S3-slow-sql-db-cpu": "慢 SQL / DB CPU 飙升",
    "S1-app-cpu-saturation": "应用 CPU 打满",
    "S8-redis-hotkey": "Redis 热 key",
    "S6-disk-inode-full": "磁盘 / inode 打满",
    "S4-memory-leak-oomkill": "内存泄漏 OOMKilled",
}
BLIND = random.choice(list(BLIND_NAMES))
print("盲测场景已注入（名称隐藏）。等待故障特征出现……\n")
try:
    start_app(BLIND, tick_period=0.05)
    # 不同场景的故障特征不同，这里用通用方式：等到任一资源指标明显异常或 5xx 上升
    wait_until(
        lambda m: (m.get("node_cpu_percent", 0) > 80
                   or m.get("db_cpu_percent", 0) > 80
                   or m.get("node_mem_percent", 0) > 90
                   or m.get("node_disk_percent", 0) > 95
                   or m.get("node_inode_percent", 0) > 95
                   or m.get("cache_latency_ms", 0) > 100
                   or m.get("container_restarts_total", 0) > 0
                   or m.get("gateway_upstream_5xx_total", 0) > 3000
                   or m.get("http_requests_total{status=\"5xx\"}", 0) > 100),
        label="任意显著异常", timeout=60)
    m = metrics()
    keys = ["node_cpu_percent", "node_mem_percent", "node_disk_percent",
            "node_inode_percent", "db_cpu_percent", "app_conn_pool_in_use",
            "app_conn_pool_size", "app_worker_busy", "cache_latency_ms",
            "cache_hit_ratio", "http_request_duration_ms_p99",
            "http_requests_total{status=\"5xx\"}", "gateway_upstream_5xx_total",
            "container_restarts_total"]
    show_metrics(m, keys)
    print("\nhealthz =", http_get("/healthz")[0])
    sample_api(20)
    print("\n再看 6 秒的变化趋势：")
    watch(seconds=6, interval=1.0, keys=keys[:10])
finally:
    stop_app()


**先写下你的判断，再运行下一格。**

In [ ]:
print("本轮盲测答案：", BLIND, "—", BLIND_NAMES[BLIND])
print("对照鉴别诊断表，检查你的推理链：是先看 RED 还是先猜场景？")


---

## 值班 SOP：从接单到复盘

### 0. 接单后 60 秒内要回答的 5 个问题

1. **影响面**：哪些客户/地域/接口？错误率多少？是否在扩大？
2. **开始时间**：几点开始？和最近发布/配置变更/扩容是否重合？
3. **现象形态**：持续还是间歇？5xx 还是超时？延迟量级？
4. **是否有止损手段**：回滚、摘除、扩容、限流、切流？
5. **需要升级吗**：影响核心交易/大面积客户时，立即升级并同步进展。

### 1. 分层排查顺序（不要跳步）

```text
接入层：网关 5xx、后端健康状态、请求到达量、连接数
  ↓
应用层：5xx、P95/P99、线程池、连接池、GC/重启
  ↓
数据层：DB CPU、慢查询、连接数、缓存延迟/命中率
  ↓
基础设施：节点 CPU/内存/磁盘/inode、DNS、网络
```

### 2. 工单记录模板

```text
【工单号】INC-YYYYMMDD-XXX
【标题】<客户> <业务> <现象>（如：订单查询间歇性 5xx）
【影响面】接口/地域/客户数/错误率/开始时间
【时间线】
  HH:MM 监控告警触发（规则名）
  HH:MM 首响客户，告知正在排查
  HH:MM 定位到 <层/组件>，证据：<指标/日志>
  HH:MM 执行止损：<动作>，效果：<指标变化>
  HH:MM 服务恢复，持续观察 N 分钟
【根因】<一句话>
【止损】<动作 + 时间>
【根治】<变更/工单 + owner + 验证方式>
【遗留风险】<是否需要扩容/重构/补监控>
```

### 3. RCA 模板（复盘不是写检讨）

```text
1. 事故概述：时间、影响面、持续时长、SLA 影响
2. 时间线：发现 → 响应 → 定位 → 止损 → 恢复（精确到分钟）
3. 根因分析：直接原因 / 根本原因（用 5 Whys，但不要停在「人为失误」）
4. 为什么没早发现：监控缺口？阈值不合理？告警被淹没？
5. 改进项：每项必须有 owner、截止时间、验证方式
   - 代码/配置修复
   - 监控/告警补齐
   - 容量/演练
   - 流程/变更约束
6. 回访客户：影响说明、改进承诺、后续观察
```

### 4. 客户沟通话术

| 阶段 | 建议说法 | 避免 |
|---|---|---|
| 首响 | 「已收到，我们正在排查。当前观察到 <现象>，影响范围是 <范围>，有进展会第一时间同步。」 | 「不清楚」「应该不是我们」 |
| 进展 | 「已定位到 <层/组件>，正在执行 <止损动作>，预计 <可验证的下一步>。」 | 未经验证的时间承诺 |
| 恢复 | 「服务已恢复，从 HH:MM 起持续观察。我们会在 <时间> 前提供 RCA 和改进计划。」 | 「已经好了」就结束 |
| 复盘 | 「根因是 <一句话>，已采取 <改进>，验证方式是 <指标>。」 | 甩锅、模糊表述 |

### 5. 变更与容量（满分工程师的分水岭）

- **变更关联**：80% 的突发故障和变更有关。先问「最近 24h 改了什么」。
- **容量水位**：CPU/内存/磁盘/inode/连接池都要有水位线和扩容预案。
- **告警质量**：每条告警都要能指向一个动作；否则就是噪音。
- **演练**：定期做故障注入，验证监控、告警、预案是否真的有效。


---

## 自评评分卡：你离「独立值班」还差什么

每个 Lab 按 5 项打分（0=不会，1=能看懂，2=能独立完成）。总分 80。

| 能力项 | 说明 |
|---|---|
| 识别 | 能在 60 秒内说出该场景最先变化的 2 个指标 |
| 鉴别 | 能说清它和最相似场景的区别，并给出排除证据 |
| 止损 | 能给出 2 个立即执行、风险可控的止损动作 |
| 根治 | 能给出 2 个防复发的工程改进，而不是「下次注意」 |
| 表达 | 能向非技术客户解释影响面，并写出 5 行以内的进展同步 |

| Lab | 识别 | 鉴别 | 止损 | 根治 | 表达 | 小计 |
|---|---|---|---|---|---|---|
| Lab 1 S5 |  |  |  |  |  |  |
| Lab 2 S7 |  |  |  |  |  |  |
| Lab 3 S2 |  |  |  |  |  |  |
| Lab 4 S3 |  |  |  |  |  |  |
| Lab 5 S1 |  |  |  |  |  |  |
| Lab 6 S8 |  |  |  |  |  |  |
| Lab 7 S6 |  |  |  |  |  |  |
| Lab 8 S4 |  |  |  |  |  |  |

**判读建议**

- **0–30**：能跑通实验，但还不能独立值班。先把「鉴别」补到 1 分以上。
- **31–55**：可以跟班处理常见告警。重点练「止损」和「表达」。
- **56–70**：可以独立值班。补容量规划、变更管理、RCA 写作。
- **71–80**：可以带新人做演练。下一步是设计监控体系与稳定性治理方案。

> 真正的「满分」不止是排障：还需要 Linux/网络/K8s/安全/成本等基础。
> 本 Notebook 覆盖的是**故障定位与应急**这一核心能力，其他能力见 `docs/` 下的架构、工作流、稳定性治理文档。


---

## Path B：Docker 全栈对照实验（可选，但面试最加分）

Path A 的指标是仿真出来的；Path B 会真的起 `HAProxy(SLB) → nginx(ingress) → app → MySQL/Redis`，
以及 Prometheus/Grafana/Alertmanager。适合练「看板 + 告警 + 多容器排障」。

### B.1 启动

```bash
# 前提：Docker Desktop 已启动，docker info 能正常输出
cd cloud-support-lab/lab
docker compose up -d
docker compose ps
```

入口：

| 组件 | 地址 |
|---|---|
| Grafana | http://localhost:3000 （admin/admin） |
| Prometheus | http://localhost:9090 |
| Alertmanager | http://localhost:9093 |
| 网关（HAProxy/SLB） | http://localhost/api/order/list |
| 应用直连 | http://localhost:8080/healthz 、/api/order/list 、/metrics |

### B.2 注入与恢复

```bash
# 注入某个场景（app 会 tick-loop 循环播放：pre → fault → post）
DRILL_SCENARIO=S5-lb-healthcheck-5xx docker compose up -d --force-recreate app

# 恢复健康
DRILL_SCENARIO= docker compose up -d --force-recreate app
```

### B.3 本 Notebook 的 8 个 Lab 与 Docker 场景一一对应

| Lab | `DRILL_SCENARIO` | 建议观察 |
|---|---|---|
| Lab 1 | `S5-lb-healthcheck-5xx` | 网关 502、`drill-haproxy` 健康检查日志、应用仍 200 |
| Lab 2 | `S7-dns-resolution-fail` | 约 30% 间歇 5xx、P99 几百毫秒 |
| Lab 3 | `S2-conn-pool-exhaustion` | 池 10/10、HikariPool 日志 |
| Lab 4 | `S3-slow-sql-db-cpu` | DB CPU、慢查询日志 |
| Lab 5 | `S1-app-cpu-saturation` | CPU 97%、线程池饱和 |
| Lab 6 | `S8-redis-hotkey` | 缓存延迟、命中率 |
| Lab 7 | `S6-disk-inode-full` | `df -h` / `df -i`、No space left |
| Lab 8 | `S4-memory-leak-oomkill` | 内存锯齿、重启计数、OOMKilled |

### B.4 S5 的真实 502 验证（重点）

S5 是最能体现「云 SLB 主动健康检查」的场景。注意 **HAProxy 需要 `fall × inter ≈ 4~5 秒` 才摘后端**，
而且要等过 pre 阶段（约 40 秒）：

```bash
# 重建容器（确保 80 端口归 haproxy，而不是旧 nginx）
docker compose down
DRILL_SCENARIO=S5-lb-healthcheck-5xx docker compose up -d --force-recreate
docker ps --format '{{.Names}}\t{{.Ports}}' | grep ':80'   # 应是 drill-haproxy

sleep 45

echo "① 走网关（期望先 200，随后大量 502）"
for i in $(seq 1 20); do curl -s -o /dev/null -w "%{http_code} " http://localhost/api/order/list; sleep 0.4; done; echo

echo "② 直连应用业务接口（期望全 200）"
for i in $(seq 1 10); do curl -s -o /dev/null -w "%{http_code} " http://localhost:8080/api/order/list; done; echo

echo "③ 直连 healthz（期望 500）"
curl -s -w "  HTTP %{http_code}\n" http://localhost:8080/healthz

echo "④ 接入层日志"
docker logs drill-haproxy --tail 20
```

正确结果：① 出现大量 502；② 全 200；③ 500。若 ① 仍全 200，按顺序检查：

1. `docker ps | grep ':80'` 是否还是 `drill-nginx`（没重建成功）。
2. 是否打太早（pre 阶段还没过）或太晚（已进入 post 恢复）。
3. HAProxy 还没累计到 `fall 2`（再等 2~4 秒）。
4. `docker logs drill-app --tail 20` 确认 `healthz=500`。

### B.5 Docker 常见问题

| 现象 | 处理 |
|---|---|
| `failed to connect to the docker API ... docker.sock` | Docker Desktop 没启动：`open -a Docker`，等图标变绿后 `docker info` |
| 镜像拉取超时 | 配置 registry mirror，或跑 `lab/scripts/pull-images.sh` |
| 80/3000/9090/3306 端口被占 | 改 compose `ports:` 左侧宿主端口 |
| 告警一直不 firing | 等过 pre 阶段与规则的 `for` 时间；看 Prometheus `/alerts` |
| `docker compose up` 卡在 mysql | 首次初始化慢/内存不足，给 Docker Desktop ≥6GB |


---

## 下一步：把「会做实验」变成「能扛值班」

### 推荐练习顺序

1. **重做一遍盲测**，但这次先写判断再开指标，训练「先假设后验证」。
2. **改一个场景参数**：打开 `lab/sim/scenarios.py`，把 S2 的 `leaked_conns` 改成 5，
   观察「池半满」时还会不会 5xx，理解阈值与容量。
3. **加一条告警**：在 `ops/prometheus/rules.yml` 里为 S5 加「后端可用实例数 < 1」的规则，
   并在 Prometheus 里验证 `pending → firing`。
4. **写一份 RCA**：用模板把任意一个 Lab 写成 1 页复盘，包含时间线与 3 个改进项。
5. **录 3 分钟讲解**：按「现象 → 分层 → 证据 → 止损 → 根治」讲一个场景，检查自己是否讲得让非技术客户听懂。

### 配套文档

| 文档 | 用途 |
|---|---|
| `README.md` / `lab/README.md` | 项目结构与运行方式 |
| `docs/01-架构与云产品映射.md` | 三层架构与云产品对应关系 |
| `docs/02-售后技术支持工作流.md` | 接单、分级、升级、回访 |
| `docs/03-故障场景库.md` | 8 个场景的完整定义 |
| `docs/04-稳定性治理方案.md` | SLO、容量、变更、告警治理 |
| `docs/05-面试讲述稿与高频题.md` | 面试表达与追问 |
| `docs/06-Mac本地实操指南.md` | Mac + Docker 全栈实操 |

> 记住一句话：**排障能力 = 固定的观察框架 + 可验证的证据链 + 可执行的止损动作**。
> 场景会忘，框架不会。
